In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
CATALOG = "clinical_trial_supply_chain"

SILVER = f"{CATALOG}.silver"

SOURCE_TABLE = f"{SILVER}.fact_shipment_events"
TARGET_TABLE = f"{SILVER}.fact_shipment_current"

In [0]:
shipment_df = spark.table(SOURCE_TABLE)

window_spec = Window.partitionBy("shipment_id") \
                    .orderBy(
                        F.col("shipment_event_timestamp").desc()
                    )

latest_df = (
    shipment_df
    .withColumn(
        "rn",
        F.row_number().over(window_spec)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [0]:
latest_df = (
    latest_df
    .withColumnRenamed(
        "shipment_status",
        "current_status"
    )
    .withColumn(
        "shipment_lead_time_days",
        F.when(
            F.col("actual_delivery_date").isNotNull(),
            F.datediff(
                F.col("actual_delivery_date"),
                F.col("expected_delivery_date")
            )
        )
    )
    .withColumn(
        "delivery_delay_days",
        F.when(
            F.col("actual_delivery_date").isNotNull(),
            F.greatest(
                F.lit(0),
                F.datediff(
                    F.col("actual_delivery_date"),
                    F.col("expected_delivery_date")
                )
            )
        )
    )
)

In [0]:
latest_df = (
    latest_df
    .withColumn(
        "shipment_date",
        F.to_date("shipment_event_timestamp")
    )
)

In [0]:
(
    latest_df.select(
        "shipment_id",
        "shipment_date",
        "source_depot_id",
        "destination_site_id",
        "study_id",
        "drug_type",
        "qty_shipped",
        "current_status",
        "expected_delivery_date",
        "actual_delivery_date",
        "shipment_lead_time_days",
        "delivery_delay_days"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)

In [0]:
display(
    spark.table(TARGET_TABLE)
)